---
title: "Statistical fine-mapping"
format:
  html:
    code-fold: false
    toc: true
  ipynb:
    toc: true
    number-sections: false
bibliography: references/references_5.bib
---

::: {.callout-note title="Important notes for this notebook" icon=false}

Discovering a genome-wide significant locus in a GWAS (`GWAS5` or `GWAS9`) is only the first step toward biological insight. Because nearby genetic variants are correlated through **Linkage Disequilibrium (LD)**, a single causal variant typically drags dozens of non-causal tagging SNPs across the significance threshold—and when a locus harbors multiple causal variants, the lead SNP (lowest $p$-value) may not even be causal at all. In this tutorial, we perform post-GWAS **statistical fine-mapping** using **SuSiE** (*Sum of Single Effects*; [Wang et al., 2020](https://doi.org/10.1111/rssb.12388); [Zou et al., 2022](https://doi.org/10.1371/journal.pgen.1010299)) to pinpoint putative causal variants and construct **95% Credible Sets**.

## Learning outcomes

- **Explain** why marginal single-SNP GWAS tests cannot distinguish true causal variants from correlated LD proxies
- **Understand** the **Sum of Single Effects (`SuSiE`)** sparse multiple regression model and how it improves upon combinatorial fine-mapping methods (`FINEMAP`, `CAVIAR`)
- **Interpret** **Posterior Inclusion Probabilities (PIPs)** and **95% Credible Sets (CS)**
- **Execute** fine-mapping in `R` using both individual-level genotypes (`susie()`) and GWAS summary statistics (`susie_rss()`)
- **Diagnose** LD reference panel mismatch when fine-mapping with summary statistics

## How to make this notebook work

* In this notebook, we will use both `bash command line` and `R` programming languages. Remember to **change the kernel** whenever you transition from one language to the other (`Kernel --> Change Kernel`) indicated by the languages' images. We will first run `Bash` commands.  
  
<img src="Images/bash.png" alt="Bash" width="80"> Choose the Bash kernel
 
:::

:::{.callout-note title="Some theory"}

### Intuition First: Why isn't the top GWAS SNP always the causal variant?

Think of a genomic region (locus) as a **group of singers standing close together on stage (in Linkage Disequilibrium, LD)** in front of a single microphone:
- **Driver vs. Passenger SNPs**: Even if **only one person is actually singing** (the true **driver** causal variant), the **passenger** SNPs standing right next to them in high LD ($r^2 \approx 0.95$) move their lips in sync and appear nearly as loud when tested one at a time in a marginal GWAS (`GWAS5` or `GWAS9`).
- **The "Sponge SNP" Trap (Synthetic Association)**: Even worse, if **two people at opposite sides of the group are singing** (two causal variants in the same locus), a non-causal **"sponge" SNP** standing in the middle that partially correlates with *both* of them ($r \approx 0.6$) soaks up signal from both and can produce a **smaller (more significant) marginal $p$-value than either true causal variant**!

**Statistical fine-mapping (`SuSiE`)** solves this by listening to all SNPs in the locus *jointly* in a single multiple-regression model and asking: *"What is the simplest sparse combination of 1, 2, or 3 variants that explains the entire association peak?"*

### Why Marginal GWAS Cannot Pinpoint Causal Variants

In a standard GWAS (`GWAS5`) or mixed-model GWAS (`GWAS9`), each SNP $j$ in a genomic region of $p$ SNPs is tested **marginally** (one at a time). If the true generative model in the locus is:

$$y = X b + \epsilon, \quad \epsilon \sim \mathcal{N}(0, \sigma_e^2 I)$$

where $X$ is the $N \times p$ standardized genotype matrix and $b = (b_1, \dots, b_p)^\top$ is a **sparse** vector of true causal effects (only a few entries $b_j \neq 0$), then the expected marginal GWAS $Z$-score vector $z = (z_1, \dots, z_p)^\top$ is:

$$\mathbb{E}[z] \approx \sqrt{N} R b, \quad \text{Cov}(z) = R$$

where $R = \frac{1}{N} X^\top X$ is the $p \times p$ local **LD correlation matrix**. Because $R$ multiplies $b$:
1. **LD Tagging**: Any non-causal SNP $k$ ($b_k = 0$) correlated with a causal SNP $j$ ($r_{jk} \neq 0$) acquires a non-zero expected test statistic $\mathbb{E}[z_k] \approx \sqrt{N} r_{jk} b_j$.
2. **Synthetic Associations**: If two causal variants $j_1$ and $j_2$ reside in the same locus, a third non-causal SNP $k$ that partially tags both can achieve a **more significant $p$-value than either true causal variant**.

### Statistical Fine-Mapping: PIPs and 95% Credible Sets

Statistical fine-mapping moves from marginal testing to **Bayesian sparse multiple regression**, fitting all $p$ SNPs in a GWAS locus simultaneously and placing a sparsity-inducing prior on $b$ (informed by the expected locus heritability). It produces two key quantities:

- **Posterior Inclusion Probability ($\text{PIP}_j$)**: The posterior probability (between `0` and `1`) that SNP $j$ itself is causal:
  $$\text{PIP}_j = \Pr(b_j \neq 0 \mid \text{Data})$$
- **95% Credible Set ($\text{CS}$)**: A minimal "bucket" of correlated SNPs $S \subseteq \{1, \dots, p\}$ that we are **at least 95% certain contains one true causal variant**.
  - *The "5 Locked Rooms" Intuition — Why do we need Credible Sets in addition to PIPs?* Suppose 1 causal variant is hiding behind one of **5 identical locked rooms** (5 SNPs in near-perfect LD, $r^2 \approx 0.99$). No statistical method can tell which of the 5 rooms holds the prize from genotype data alone, so each SNP receives $\text{PIP}_j \approx 0.20$. Looking only for $\text{PIP}_j > 0.90$ would miss the signal completely—whereas a **95% Credible Set** groups those 5 rooms together ($\sum \text{PIP} \ge 0.95$) and tells you: *"One of these 5 variants is causal with 95% certainty, and the other 195 SNPs in the locus can be ruled out."*

### How `SuSiE` (Sum of Single Effects) Works

Earlier fine-mapping tools (`CAVIAR`, `FINEMAP`) had to evaluate discrete combinations of causal configurations across $\{0, 1\}^p$, which becomes computationally expensive as the number of causal variants $L$ grows and makes it difficult to separate multiple independent signals into distinct credible sets.

**`SuSiE`** ([Wang et al., 2020](https://doi.org/10.1111/rssb.12388)) solves this elegantly by writing the sparse effect vector $b$ as a **Sum of $L$ Single Effects**:

$$b = \sum_{l=1}^{L} b^{(l)}, \quad b^{(l)} = \gamma^{(l)} \beta^{(l)}$$

where each "single-effect" vector $b^{(l)}$ has **at most one non-zero entry**:
- $\gamma^{(l)} \in \{0, 1\}^p$ is a one-hot indicator vector drawn from $\text{Multinomial}(1, \pi)$ specifying which single SNP is causal for effect $l$.
- $\beta^{(l)} \sim \mathcal{N}(0, \sigma_{0l}^2)$ is the scalar effect size for that causal SNP.

Using **Iterative Bayesian Stepwise Selection (IBSS)**, `SuSiE` cycles through $l = 1, \dots, L$, fitting a simple single-SNP Bayesian regression on the partial residuals after removing the other $L - 1$ effects. Each component $l$ yields posterior probabilities $\alpha_l = (\alpha_{l1}, \dots, \alpha_{lp})$ that directly define the $l$-th **95% Credible Set** (by sorting $\alpha_{lj}$ in descending order until their cumulative sum reaches $0.95$), and the overall PIP for SNP $j$ is simply:

$$\text{PIP}_j = 1 - \prod_{l \in \mathcal{L}_{\text{non-null}}} (1 - \alpha_{lj})$$

| Feature | LD Clumping (`PLINK --clump`) | `FINEMAP` / `CAVIAR` | `SuSiE` / `SuSiE-RSS` (`susieR`) |
|---|---|---|---|
| **Statistical Framework** | Heuristic greedy $r^2$ + $p$-value thresholding | Combinatorial / Shotgun Stochastic Search | Variational Bayes (**Sum of Single Effects**) |
| **Outputs PIPs?** | No | Yes | **Yes** |
| **Outputs Separate Credible Sets per Signal?** | No | Post-hoc approximations | **Yes — one CS per single-effect component $l$** |
| **Input Data Supported** | Summary stats + LD reference | Summary stats + LD matrix $R$ | **Both Individual-level (`susie`) & Summary stats (`susie_rss`)** |

> **Quick Concept Check (from Lecture 10)**:
> - *If `SuSiE` outputs a 95% Credible Set (`CS1`) of 5 SNPs in perfect LD ($r^2 = 1.00$), each with $\text{PIP} = 0.20$, is the locus a false positive?* **No!** We are $\ge 95\%$ certain **1 true causal signal** is inside that bucket, split equally across 5 indistinguishable "locked rooms" ($5 \times 0.20 = 1.00$).
> - *When can the #1 top GWAS SNP (smallest $p$-value) receive $\text{PIP} \approx 0.00$ in `SuSiE`?* When it is a non-causal **"sponge" SNP** that partially tags two true causal variants!

:::

# Preparing Locus Data for Fine-Mapping

### Extracting a GWAS Locus and Local LD Matrix ($R$)

In practice, after running a genome-wide scan (`GWAS5` or `GWAS9`), we extract a $\sim 500\text{ kb}$–$1\text{ Mb}$ window around each genome-wide significant peak.

<img src="Images/bash.png" alt="Bash" width="40"> Switch to the Bash kernel.

Let's link the `Data` folder, create `Results/GWAS10`, and extract a 200-SNP locus on Chromosome 1 from `Data/EUR.QC` using `PLINK` to generate two inputs for fine-mapping:
- `--extract <file>`: Restricts `PLINK` to the 200 SNPs in our locus.
- `--recode A`: Exports the individual-level additive genotype matrix (`locus_geno.raw`), where each cell is `0`, `1`, or `2` copies of the minor allele (used by `susie()`).
- `--r square`: Computes the $200 \times 200$ signed SNP-by-SNP LD correlation matrix $R$ (`locus_ld.ld`, used by `susie_rss()`).

In [ ]:
ln -sf ../Data
mkdir -p Results/GWAS10

# Select a 200-SNP window on Chromosome 1
awk '$1 == 1 {print $2}' Data/EUR.QC.bim | head -n 500 | tail -n 200 > Results/GWAS10/locus.snps

# Save SNP coordinates (SNP, CHR, BP) for locus plotting
awk '$1 == 1 {print $2, $1, $4}' Data/EUR.QC.bim | head -n 500 | tail -n 200 > Results/GWAS10/locus.info

# 1. Export additive genotype matrix (0/1/2 dosages) for individual-level SuSiE
plink \
  --bfile Data/EUR.QC \
  --extract Results/GWAS10/locus.snps \
  --recode A \
  --out Results/GWAS10/locus_geno

# 2. Compute the p x p local LD correlation matrix (R) for summary-statistics SuSiE-RSS
plink \
  --bfile Data/EUR.QC \
  --extract Results/GWAS10/locus.snps \
  --r square \
  --out Results/GWAS10/locus_ld

echo "Extracted locus SNPs: $(wc -l < Results/GWAS10/locus.snps)"

### Simulating a Multi-Causal Locus and Running Marginal GWAS in R

<img src="Images/R.png" alt="R" width="40"> Switch to the R kernel.

Now switch to the **R kernel**. Let's load `susieR`, read our 200-SNP genotype matrix $X$ and LD matrix $R$, and simulate a quantitative phenotype driven by **two causal variants** in this region (`causal_idx = c(65, 140)`), which together explain $h^2_{\text{locus}} = 12\%$ of phenotypic variance.

In [ ]:
# Setup to avoid long messages and plot on screen
options(warn = -1)
options(jupyter.plot_mimetypes = 'image/png')
options(repr.plot.width = 9, repr.plot.height = 7)

if (!requireNamespace("susieR", quietly = TRUE)) {
  install.packages("susieR", repos = "https://cloud.r-project.org")
}
suppressMessages(suppressWarnings(library(susieR)))
suppressMessages(suppressWarnings(library(ggplot2)))

# 1. Load genotype matrix X (N x p), SNP coordinates, and LD matrix R (p x p)
raw_geno   <- read.table("Results/GWAS10/locus_geno.raw", header = TRUE)
locus_info <- read.table("Results/GWAS10/locus.info", header = FALSE,
                         col.names = c("SNP", "CHR", "BP"))
R <- as.matrix(read.table("Results/GWAS10/locus_ld.ld", header = FALSE))

# Columns 7..ncol in PLINK .raw are the SNP dosages
X <- as.matrix(raw_geno[, 7:ncol(raw_geno)])
colnames(X) <- locus_info$SNP

# Mean-impute any sporadic missing genotypes
for (j in seq_len(ncol(X))) {
  X[is.na(X[, j]), j] <- mean(X[, j], na.rm = TRUE)
}

n <- nrow(X)
p <- ncol(X)
cat("Sample size (N):", n, "| Locus SNPs (p):", p, "\n")

In [ ]:
# 2. Simulate a phenotype with 2 true causal variants in the locus
set.seed(2025)
causal_idx <- c(65, 140)
b_true <- rep(0, p)
b_true[causal_idx] <- c(0.45, -0.45)

# Standardize X to mean 0, variance 1 for clean effect simulation
X_std <- scale(X)
g_locus <- as.numeric(X_std %*% b_true)
# Calibrate residual noise so the locus explains ~12% of phenotypic variance
sigma_e <- sqrt(var(g_locus) * (1 - 0.12) / 0.12)
y <- g_locus + rnorm(n, mean = 0, sd = sigma_e)

# 3. Compute marginal single-SNP GWAS summary statistics (as PLINK/LDAK-KVIK would output)
uni_reg <- univariate_regression(X, y)
z_scores <- uni_reg$betahat / uni_reg$sebetahat
p_values <- 2 * pnorm(-abs(z_scores))

cat("True causal SNPs:", locus_info$SNP[causal_idx], "\n")
cat("Top marginal GWAS hit:", locus_info$SNP[which.min(p_values)],
    "(p =", signif(min(p_values), 3), ")\n")

# Individual-Level Fine-Mapping with `susie()`

When we have access to individual-level genotypes $X$ and phenotype $y$, we run `susie(X, y, L = 10)`:
- `L = 10` sets the **upper bound** on the number of causal signals in the locus. `SuSiE` automatically prunes away unneeded single-effect components by shrinking their prior variance to zero, so setting `L = 10` easily accommodates 1, 2, or more causal variants.
- `summary(fitted_ind)$cs` prints a table of discovered **95% Credible Sets**, where:
  - `cs`: Index of the Credible Set (`1`, `2`, ...)
  - `cs_log10bf`: $\log_{10}(\text{Bayes Factor})$ measuring the strength of evidence for this causal signal
  - `cs_avg_r2` / `cs_min_r2`: Average and minimum pairwise LD ($r^2$) among SNPs inside the Credible Set (purity)
  - `variable`: Indices of the SNPs belonging to this 95% Credible Set

In [ ]:
# Fit individual-level SuSiE with up to L = 10 single effects
fitted_ind <- susie(X, y, L = 10)

# Print summary of discovered 95% Credible Sets
print(summary(fitted_ind)$cs)

# Check whether our two true causal variants (indices 65 and 140) are captured in the Credible Sets
cs_list <- fitted_ind$sets$cs
for (cs_name in names(cs_list)) {
  snps_in_cs <- cs_list[[cs_name]]
  captured <- intersect(snps_in_cs, causal_idx)
  cat(cs_name, "contains", length(snps_in_cs), "SNP(s):", paste(snps_in_cs, collapse = ", "),
      "| True causal index captured:", ifelse(length(captured) > 0, captured, "None"), "\n")
}

Now let's visualize the **Marginal GWAS Locus Plot** (`-log10(p)`) right above the **`SuSiE` Fine-Mapping Plot** (`PIP`), highlighting the discovered 95% Credible Sets and the two true causal variants.

In [ ]:
# Assign Credible Set membership for plotting
cs_membership <- rep("None", p)
for (cs_name in names(cs_list)) {
  cs_membership[cs_list[[cs_name]]] <- paste0("Credible Set ", cs_name)
}

plot_df <- data.frame(
  Index    = 1:p,
  BP_Mb    = locus_info$BP / 1e6,
  NegLogP  = -log10(p_values),
  PIP      = fitted_ind$pip,
  CS       = cs_membership,
  IsCausal = ifelse(1:p %in% causal_idx, "True Causal SNP", "Non-Causal SNP")
)

par(mfrow = c(2, 1), mar = c(4, 4.5, 2.5, 1))
# Panel 1: Marginal GWAS -log10(p)
plot(plot_df$BP_Mb, plot_df$NegLogP,
     pch = ifelse(plot_df$IsCausal == "True Causal SNP", 17, 19),
     col = ifelse(plot_df$IsCausal == "True Causal SNP", "tomato", "steelblue"),
     cex = ifelse(plot_df$IsCausal == "True Causal SNP", 1.6, 1.0),
     xlab = "", ylab = expression(-log[10](italic(p))),
     main = "Marginal GWAS Association Signal Across Locus (Confounded by LD)")
abline(h = -log10(0.05 / p), lty = 2, col = "grey40")
legend("topright", legend = c("True Causal SNP", "Tagging / Null SNP"),
       col = c("tomato", "steelblue"), pch = c(17, 19), bty = "n")

# Panel 2: SuSiE Posterior Inclusion Probabilities (PIP) & Credible Sets
susie_plot(fitted_ind, y = "PIP", b = b_true,
           main = "SuSiE Fine-Mapping: Posterior Inclusion Probabilities (PIPs) & 95% Credible Sets")

Look at the difference between the two panels:
- In the **top panel (Marginal GWAS)**, many neighboring SNPs show elevated $-\log_{10}(p)$ values due to LD with the two causal variants.
- In the **bottom panel (`SuSiE` PIPs)**, conditioning on all SNPs jointly collapses the signal into **two distinct 95% Credible Sets** (shown by the colored circles), each containing one of the true causal variants (red dots)!

# Summary-Statistics Fine-Mapping with `susie_rss()`

In most consortium GWAS studies, individual-level genotypes $X$ are not shared publicly. Instead, we only have:
1. Marginal GWAS summary statistics ($Z$-scores `z_scores`, or $\hat{\beta}$ and $\text{SE}$)
2. The local LD correlation matrix $R$ (`p x p`), ideally computed from the discovery cohort (in-sample LD) or a large ancestry-matched reference panel
3. The GWAS sample size `n`

Let's run **`susie_rss()`** using only `z_scores`, `R`, and `n`, and inspect the resulting 95% Credible Sets.

In [ ]:
# Run SuSiE-RSS using only summary Z-scores, LD matrix R, and sample size n
fitted_rss <- susie_rss(z = z_scores, R = R, n = n, L = 10)

# Inspect the 95% Credible Sets from summary statistics
print(summary(fitted_rss)$cs)

:::{.callout-important title="Stop - Read - Solve"}

- **Q1. Compare the PIPs estimated from individual-level data (`fitted_ind$pip`) with the PIPs estimated from summary statistics + in-sample LD (`fitted_rss$pip`). Create a scatter plot in `ggplot2` and compute their Pearson correlation.**
- **Q2. For each discovered 95% Credible Set in `fitted_ind$sets$cs`, print (a) the number of SNPs in the Credible Set, (b) the maximum single-SNP PIP inside the set, and (c) the sum of PIPs across all SNPs in the set. Why can the sum of PIPs in a Credible Set exceed 0.95 even when no individual SNP has $\text{PIP} > 0.95$?**
- **Q3. What happens when the LD reference matrix $R$ comes from a small or mismatched reference panel? Compute an imprecise LD matrix `R_small <- cor(X[1:60, ])` from only 60 individuals, run `estimate_s_rss(z = z_scores, R = R_small, n = n)` to check the `SuSiE-RSS` LD consistency diagnostic $s$, and fit `susie_rss(z = z_scores, R = R_small, n = n, L = 10)`. How does LD mismatch affect fine-mapping?**

Hint: Compare `estimate_s_rss(z = z_scores, R = R, n = n)` (in-sample LD, where $s \approx 0$) vs. `estimate_s_rss(z = z_scores, R = R_small, n = n)` (where larger $s$ indicates inconsistency between $z$ and $R$).
:::

In [ ]:
# Write your code here


:::{.callout-tip title="Solution" collapse="true"}

In `R`, we compare `fitted_ind$pip` and `fitted_rss$pip`, inspect the PIPs within each 95% Credible Set, and evaluate the LD diagnostic $s$ when using a small reference panel:

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4.5)

# Q1. Compare individual-level SuSiE PIPs vs. summary-statistics SuSiE-RSS PIPs
pip_comp <- data.frame(
  SNP      = locus_info$SNP,
  PIP_ind  = fitted_ind$pip,
  PIP_rss  = fitted_rss$pip,
  IsCausal = ifelse(1:p %in% causal_idx, "True Causal SNP", "Non-Causal SNP")
)

cat("Pearson correlation between susie() and susie_rss() PIPs:",
    round(cor(pip_comp$PIP_ind, pip_comp$PIP_rss), 5), "\n\n")

p_pip <- ggplot(pip_comp, aes(x = PIP_ind, y = PIP_rss, color = IsCausal)) +
  geom_point(size = 2.5, alpha = 0.85) +
  geom_abline(intercept = 0, slope = 1, linetype = "dashed") +
  scale_color_manual(values = c("True Causal SNP" = "tomato", "Non-Causal SNP" = "steelblue")) +
  theme_bw(base_size = 13) +
  labs(
    title = "SuSiE Individual-Level vs. Summary-Statistics (SuSiE-RSS) PIPs",
    x = "Individual-Level PIP (susie)",
    y = "Summary-Statistics PIP (susie_rss with in-sample LD)",
    color = "Variant Class"
  )

show(p_pip)

# Q2. Inspect PIPs inside each 95% Credible Set
for (cs_name in names(fitted_ind$sets$cs)) {
  idx <- fitted_ind$sets$cs[[cs_name]]
  cat(cs_name, "| Size:", length(idx),
      "| Max PIP:", round(max(fitted_ind$pip[idx]), 3),
      "| Sum of PIPs:", round(sum(fitted_ind$pip[idx]), 3), "\n")
}

# Q3. Diagnose and evaluate noisy/mismatched LD reference matrix
R_small <- cor(X[1:60, ])
R_small[is.na(R_small)] <- 0
diag(R_small) <- 1

s_insample <- estimate_s_rss(z = z_scores, R = R, n = n)
s_small    <- estimate_s_rss(z = z_scores, R = R_small, n = n)
cat("\nLD diagnostic s (In-sample R):   ", round(s_insample, 4), "\n")
cat("LD diagnostic s (Small panel R): ", round(s_small, 4), "\n")

fitted_rss_small <- susie_rss(z = z_scores, R = R_small, n = n, L = 10)
cat("Number of Credible Sets with noisy LD:", length(fitted_rss_small$sets$cs), "\n")

- **Q1**: When supplied with the exact **in-sample LD matrix $R$**, `susie_rss()` recovers PIPs and 95% Credible Sets that are virtually identical ($r > 0.999$) to having the full individual-level genotype matrix $X$.
- **Q2**: When a causal variant is in high LD with one or more proxy SNPs, the posterior probability is split across the correlated SNPs in the credible set, so no single SNP may exceed $\text{PIP} > 0.95$, while their **sum** exceeds $0.95$. Always report the full **95% Credible Set** alongside individual PIPs.
- **Q3**: When the LD reference matrix $R$ does not match the GWAS discovery sample (reflected by an elevated diagnostic parameter $s$ in `estimate_s_rss`), `SuSiE-RSS` cannot accurately cancel out LD tagging and may split signals or produce false-positive credible sets. Always use in-sample LD or a large, ancestry-matched reference panel.

:::

---

### Conclusion

In this tutorial, we have learned how statistical fine-mapping with `SuSiE` resolves broad, LD-confounded GWAS peaks into prioritized **Posterior Inclusion Probabilities (PIPs)** and **95% Credible Sets** using both individual-level genotypes (`susie()`) and GWAS summary statistics (`susie_rss()`). Notice how `susie_rss()` pairs marginal GWAS summary statistics (`z`) with an LD correlation matrix (`R`) to undo LD confounding within a single locus. In the final session (`GWAS11`), we scale this exact same principle—combining **GWAS summary statistics** with an **external LD reference panel** (`ldak --calc-cors`) and per-SNP heritability priors (`sumher_ldak.ind.hers` from `GWAS8`)—across the entire genome to construct **advanced Polygenic Risk Scores (PRS)**.

:::{.callout-note title="Other take-home messages"}

- **Multiple Causal Variants Are Common**: A single GWAS locus frequently contains multiple distinct causal signals ($L > 1$). `SuSiE` separates these signals into distinct 95% Credible Sets without requiring combinatorial search.
- **Report Both PIPs and 95% Credible Sets**: High LD among candidate variants splits the PIP across proxies; the 95% Credible Set captures the complete group of variants needed to guarantee $\ge 95\%$ coverage of the causal signal.
- **LD Reference Quality Is Critical for Summary-Statistics Fine-Mapping**: When using `susie_rss()`, always match the ancestry and sample size of the LD matrix $R$ to the GWAS summary statistics and check `estimate_s_rss()` for potential LD mismatch.

:::